# Run 4b - TF-IDF + Word2Vec + SVM


In [ ]:
!pip install -q pandas scikit-learn gensim scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.9 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.svm import SVC
from google.colab import files

print("Imports OK")

Imports OK


In [ ]:
# Upload train.csv ET test.csv
uploaded = files.upload()

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train : {len(train)} recettes")
print(f"Test  : {len(test)} recettes")

assert len(train) == 12473, f"ERREUR train incomplet : {len(train)} lignes au lieu de 12473"
assert len(test)  == 1388,  f"ERREUR test incomplet : {len(test)} lignes au lieu de 1388"

print("Dataset OK")
print(train["type"].value_counts())

y_train = train["type"]
y_test  = test["type"]

Saving test.csv to test.csv
Saving train.csv to train.csv
Train : 12473 recettes
Test  : 1388 recettes
Dataset OK
type
Plat principal    5802
Dessert           3762
Entrée            2909
Name: count, dtype: int64


In [ ]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zàâäéèêëîïôöùûüçœæ\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def build_text(df, columns):
    return df[columns].fillna("").astype(str).agg(" ".join, axis=1)

# Texte complet titre + ingredients + recette
x_train_text = build_text(train, ["titre", "ingredients", "recette"]).apply(preprocess)
x_test_text  = build_text(test,  ["titre", "ingredients", "recette"]).apply(preprocess)

print("Preprocessing OK")
print(f"Exemple : {x_train_text.iloc[0][:200]}")

Preprocessing OK
Exemple : feuilleté de saumon et de poireau sauce aux crevettes gros pavé de saumon g de crevettes décortiquées poireaux moyens oignon pâte feuilletée cl de crème liquide épaisse un peu de vin blanc citron jaun


In [ ]:
# TF-IDF avec meilleurs params (Grid Search Run 2)
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True, min_df=2)

x_train_tfidf = tfidf.fit_transform(x_train_text)
x_test_tfidf  = tfidf.transform(x_test_text)

print(f"TF-IDF train : {x_train_tfidf.shape}")
print(f"TF-IDF test  : {x_test_tfidf.shape}")

TF-IDF train : (12473, 10000)
TF-IDF test  : (1388, 10000)


In [ ]:
# Entrainement Word2Vec sur le corpus train
sentences = [simple_preprocess(t) for t in x_train_text]

w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    seed=42
)
print(f"Word2Vec entraine - vocab : {len(w2v.wv)} mots")

Word2Vec entraine - vocab : 11867 mots


In [ ]:
def doc_vector(text, model):
    tokens = simple_preprocess(str(text))
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0).astype(np.float32) if vecs else np.zeros(model.vector_size, dtype=np.float32)

print("Vectorisation Word2Vec train...")
x_train_w2v = np.array([doc_vector(t, w2v) for t in x_train_text], dtype=np.float32)
print("Vectorisation Word2Vec test...")
x_test_w2v  = np.array([doc_vector(t, w2v) for t in x_test_text],  dtype=np.float32)

print(f"Word2Vec train : {x_train_w2v.shape}")
print(f"Word2Vec test  : {x_test_w2v.shape}")

Vectorisation Word2Vec train...
Vectorisation Word2Vec test...
Word2Vec train : (12473, 100)
Word2Vec test  : (1388, 100)


In [ ]:
# Combinaison TF-IDF + Word2Vec
x_train_combined = hstack([x_train_tfidf, csr_matrix(x_train_w2v)], format="csr")
x_test_combined  = hstack([x_test_tfidf,  csr_matrix(x_test_w2v)],  format="csr")

print(f"Combined train : {x_train_combined.shape}")
print(f"Combined test  : {x_test_combined.shape}")

Combined train : (12473, 10100)
Combined test  : (1388, 10100)


In [ ]:
print("Entrainement SVM...")
svm = SVC(kernel="linear", C=1.0, random_state=42)
svm.fit(x_train_combined, y_train)
y_pred = svm.predict(x_test_combined)

f1  = f1_score(y_test, y_pred, average="macro")
acc = accuracy_score(y_test, y_pred)

print("=== Run 4b - TF-IDF + Word2Vec + SVM ===")
print(classification_report(y_test, y_pred))
print(f"Accuracy : {acc:.4f}")
print(f"F1 macro : {f1:.4f}")

Entrainement SVM...
=== Run 4b - TF-IDF + Word2Vec + SVM ===
                precision    recall  f1-score   support

       Dessert       0.99      0.99      0.99       407
        Entrée       0.77      0.70      0.73       337
Plat principal       0.86      0.89      0.87       644

      accuracy                           0.87      1388
     macro avg       0.87      0.86      0.87      1388
  weighted avg       0.87      0.87      0.87      1388

Accuracy : 0.8746
F1 macro : 0.8653


In [ ]:
print("=" * 55)
print("RECAP GENERAL - tous les runs")
print("=" * 55)
recap = [
    ("Run 1  Baseline",                   0.211),
    ("Run 2  TF-IDF + SVM",               0.863),
    ("Run 3a Word2Vec + SVM",              0.844),
    ("Run 3b CamemBERT + SVM",             0.708),
    ("Run 4b TF-IDF + Word2Vec + SVM",     round(f1, 3)),
]
for name, score in recap:
    bar = chr(9608) * int(score * 30)
    print(f"{name:<42} {score:.3f}  {bar}")

pd.Series(y_pred, name="prediction").to_csv("predictions_run4b_w2v.csv", index=False)
files.download("predictions_run4b_w2v.csv")

RECAP GENERAL - tous les runs
Run 1  Baseline                            0.211  ██████
Run 2  TF-IDF + SVM                        0.863  █████████████████████████
Run 3a Word2Vec + SVM                      0.844  █████████████████████████
Run 3b CamemBERT + SVM                     0.708  █████████████████████
Run 4b TF-IDF + Word2Vec + SVM             0.865  █████████████████████████


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>